# S4 — 特徵工程思維：從觀察到創造

> **時間**：2 小時（講授 70min + 課堂練習 40min + QA 10min）  
> **資料**：`datasets/titanic/train.csv`（講授）、`datasets/spaceship-titanic/train.csv`（練習）  
> **學完能幹嘛**：從原始欄位中創造出有預測力的新特徵，不靠模型也能大幅提升分析深度

---

## 為什麼特徵工程是 Kaggle 致勝關鍵？

Kaggle 金牌選手的名言：「特徵工程決定上限，模型只是逼近上限。」

同一份 Titanic 資料，原始 12 欄位丟進模型 → 準確率 78%。  
加上精心設計的 6 個新特徵 → 準確率 85%。模型完全一樣。

**一句話口訣：原始欄位是食材，特徵工程是料理 — 同一塊豆腐，紅燒和涼拌完全不同價值。**

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns

df = pd.read_csv('datasets/titanic/train.csv')
print(f'Titanic: {df.shape[0]} rows × {df.shape[1]} columns')
print(f'原始欄位: {list(df.columns)}')

---
## 核心觀念 1／3 — 組合特徵：1 + 1 > 2

把 2 個以上的欄位**組合**成新欄位，捕捉原本分開看不到的關係。

| 組合方式 | 範例 | 效果 |
|:---------|:-----|:-----|
| 相加 | SibSp + Parch + 1 = FamilySize | 家庭大小 |
| 二值化 | FamilySize == 1 → IsAlone | 是否獨自旅行 |
| 相乘 | 面積 × 樓層 = 總面積 | 交互效果 |

In [ ]:
# 特徵 1：FamilySize
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
print('FamilySize 分布：')
print(df['FamilySize'].value_counts().sort_index())

# 立即驗證：FamilySize vs Survived
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x='FamilySize', y='Survived', data=df, ax=ax)
ax.set_title('存活率 by FamilySize', fontsize=14)
ax.set_ylabel('存活率')
plt.tight_layout()
plt.show()

print('→ 2-4 人家庭存活率最高，獨行或大家庭存活率低')

In [ ]:
# 特徵 2：IsAlone
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

print('IsAlone vs Survived：')
print(pd.crosstab(df['IsAlone'], df['Survived'], normalize='index').round(3))

fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(x='IsAlone', y='Survived', data=df, ax=ax,
            palette=['#66b2ff', '#ff9999'])
ax.set_title('存活率：獨行 vs 有同伴', fontsize=14)
ax.set_xticklabels(['有同伴 (0)', '獨行 (1)'])
plt.tight_layout()
plt.show()

print('→ 獨行者存活率 30.4%，有同伴者 50.6% — 差距顯著')

**口訣**：做完新特徵，馬上用 barplot/crosstab 驗證它跟 target 有沒有區分度 — 沒用的特徵不要留。

---
## 核心觀念 2／3 — 文字拆解：從字串中挖金礦

很多「看起來沒用」的文字欄位，其實藏著寶貴資訊。

| 原始欄位 | 拆解方式 | 新特徵 |
|:---------|:---------|:-------|
| Name | `str.extract(r' ([A-Za-z]+)\.')` | Title（Mr, Mrs, Miss...） |
| Cabin | `str[0]` | Deck（A, B, C...） |
| Ticket | `str.split().str[0]` | Ticket prefix |

In [ ]:
# 特徵 3：從 Name 拆出 Title
df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.')
print('Title 分布：')
print(df['Title'].value_counts())

In [ ]:
# 合併稀有 Title
rare_titles = df['Title'].value_counts()
rare_titles = rare_titles[rare_titles < 10].index
df['Title'] = df['Title'].replace(rare_titles, 'Rare')

print('合併後 Title 分布：')
print(df['Title'].value_counts())

In [ ]:
# 驗證：Title vs Survived
fig, ax = plt.subplots(figsize=(8, 5))
title_survival = df.groupby('Title')['Survived'].mean().sort_values(ascending=False)
title_survival.plot(kind='bar', color='steelblue', edgecolor='white', ax=ax)
ax.set_title('存活率 by Title', fontsize=14)
ax.set_ylabel('存活率')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

print('→ Mrs/Miss 存活率最高（女性）、Mr 最低（成年男性）')
print('→ Master 代表年幼男孩，存活率也較高 — Title 是強特徵！')

In [ ]:
# 特徵 4：從 Cabin 拆出 Deck
df['Deck'] = df['Cabin'].str[0]  # 取第一個字母
print('Deck 分布（含 NaN）：')
print(df['Deck'].value_counts(dropna=False))

# 驗證
deck_survival = df.groupby('Deck')['Survived'].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 5))
deck_survival.plot(kind='bar', color='steelblue', edgecolor='white', ax=ax)
ax.set_title('存活率 by Deck', fontsize=14)
ax.set_ylabel('存活率')
plt.tight_layout()
plt.show()

print('→ 不同 Deck 存活率差異大（B, D, E 較高）— 可能跟救生艇位置有關')

**口訣**：`str.extract` 用正則拆、`str[0]` 取首字、`str.split` 切分 — 文字欄位是金礦，不要直接丟掉。

> ### 💡 知識補給站 — 正則表達式速查
> 
> 特徵工程常用的正則：
> 
> | 模式 | 含義 | 範例 |
> |:-----|:-----|:-----|
> | `[A-Za-z]+` | 一個以上英文字母 | `Mr`, `Mrs` |
> | `\d+` | 一個以上數字 | `123` |
> | `\.` | 句點（需跳脫） | `.` |
> | `(...)` | 擷取群組 | 只取括號內的部分 |
> | `\s` | 空白字元 | 空格、tab |
> 
> `str.extract(r' ([A-Za-z]+)\.')` 的意思：
> 找「空格 + 一個以上英文字母 + 句點」這個模式，**只擷取英文字母部分**。
> 
> *延伸關鍵字：regex, str.extract, str.contains, str.findall*

---
## 核心觀念 3／3 — 分箱 (Binning) 與交互特徵

| 技巧 | 方法 | 何時用 |
|:-----|:-----|:-------|
| 等頻分箱 | `pd.qcut(col, q=4)` | 讓每組人數相等 |
| 等距分箱 | `pd.cut(col, bins=4)` | 讓每組範圍相等 |
| 自訂分箱 | `pd.cut(col, bins=[0,12,18,...])` | 有領域知識時用 |
| 交互特徵 | 兩個欄位組合成新欄位 | 捕捉交互效果 |

In [ ]:
# 分箱比較：qcut vs cut
df_age = df.dropna(subset=['Age']).copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# qcut：等頻
df_age['Age_qcut'] = pd.qcut(df_age['Age'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
sns.barplot(x='Age_qcut', y='Survived', data=df_age, ax=axes[0])
axes[0].set_title('qcut (等頻) — 存活率 by 年齡組', fontsize=12)

# cut：自訂
df_age['Age_cut'] = pd.cut(df_age['Age'], bins=[0, 12, 18, 35, 60, 80],
                           labels=['Child', 'Teen', 'Young', 'Adult', 'Senior'])
sns.barplot(x='Age_cut', y='Survived', data=df_age, ax=axes[1])
axes[1].set_title('cut (自訂) — 存活率 by 年齡組', fontsize=12)

plt.tight_layout()
plt.show()

print('→ cut 更有解釋性（兒童/青少年/青年/成人/長者）')
print('→ qcut 保證每組人數平衡，模型訓練時較穩定')

In [ ]:
# 交互特徵：Sex × Pclass
df['Sex_Pclass'] = df['Sex'] + '_' + df['Pclass'].astype(str)

fig, ax = plt.subplots(figsize=(10, 5))
order = ['female_1', 'female_2', 'female_3', 'male_1', 'male_2', 'male_3']
sns.barplot(x='Sex_Pclass', y='Survived', data=df, ax=ax, order=order)
ax.set_title('存活率 by Sex × Pclass 交互特徵', fontsize=14)
ax.set_ylabel('存活率')
plt.tight_layout()
plt.show()

print('→ female_1 和 female_2 存活率 > 90%，male_3 只有 ~14%')
print('→ 交互特徵比單獨的 Sex 或 Pclass 有更細緻的區分度')

In [ ]:
# 用 pivot_table 驗證交互效果
pivot = pd.pivot_table(df, values='Survived', index='Sex',
                       columns='Pclass', aggfunc='mean')
print('Survival Rate Pivot Table (Sex × Pclass)：')
print(pivot.round(3))

**口訣**：`qcut` 等頻、`cut` 自訂、交互特徵 = 欄位A + '_' + 欄位B — 每做完一個特徵，馬上用 barplot 驗證。

---
## 實務 Case — Titanic 特徵工程全攻略

把今天學的全部串起來：從原始 12 欄位，建立 6 個新特徵。

In [ ]:
# 特徵工程總覽
new_features = ['FamilySize', 'IsAlone', 'Title', 'Deck', 'Sex_Pclass']
print('新特徵一覽：')
for feat in new_features:
    if feat in df.columns:
        print(f'  {feat}: {df[feat].nunique()} 個唯一值, 缺值 {df[feat].isnull().sum()}')

print(f'\n原始欄位：12 → 加上新特徵後：{len(df.columns)}')

---
## 課堂練習（40 min）

用 **Spaceship Titanic** 資料集練習特徵工程。

### 🟢 送分題

1. 讀取 `datasets/spaceship-titanic/train.csv`，跑 `info()` 看欄位
2. 建立 `TotalSpending` 特徵 = `RoomService + FoodCourt + ShoppingMall + Spa + VRDeck` 的加總
3. 用 barplot 比較 `Transported=True` vs `False` 的平均 `TotalSpending`

In [ ]:
# TODO: 你的程式碼

### 🟡 核心題

Spaceship Titanic 的 `Cabin` 欄位格式是 `deck/num/side`（例如 `B/0/P`）。

1. 用 `str.split('/')` 拆成 3 個新欄位：`Cabin_deck`, `Cabin_num`, `Cabin_side`
2. 畫出 `Transported` rate by `Cabin_deck` 的 barplot
3. 畫出 `Transported` rate by `Cabin_side`（P=Port, S=Starboard）的 barplot

In [ ]:
# TODO: 你的程式碼

### 🔴 挑戰題

1. 建立 `IsVIP` 特徵 = `VIP` 欄位轉成 0/1
2. 建立 `SpendingCategory` = 用 `pd.qcut(TotalSpending, q=3, labels=['Low','Mid','High'])` 分箱
   （注意 TotalSpending 有很多 0，可能需要處理）
3. 建立交互特徵 `Deck_Side` = `Cabin_deck` + '_' + `Cabin_side`
4. 在一張 2×2 的子圖中，畫出這 4 個新特徵各自對 `Transported` 的影響
5. 哪個新特徵的區分度最高？

In [ ]:
# TODO: 你的程式碼

---
## 小結與速查表

| 想做什麼 | 怎麼寫 |
|:---------|:-------|
| 欄位相加 | `df['new'] = df['a'] + df['b']` |
| 二值化 | `df['flag'] = (df['col'] == val).astype(int)` |
| 正則拆解 | `df['col'].str.extract(r'(pattern)')` |
| 取首字 | `df['col'].str[0]` |
| 字串分割 | `df['col'].str.split('/').str[0]` |
| 等頻分箱 | `pd.qcut(df['col'], q=4, labels=[...])` |
| 自訂分箱 | `pd.cut(df['col'], bins=[...], labels=[...])` |
| 交互特徵 | `df['a'] + '_' + df['b'].astype(str)` |
| 快速驗證 | `df.groupby('new_feat')['target'].mean()` |
| 交叉驗證 | `pd.crosstab(df['feat'], df['target'], normalize='index')` |
| Pivot 驗證 | `pd.pivot_table(df, values='target', index='a', columns='b')` |

**下節預告 S5**：我們已經學會 First Look → Missing → Distribution → Feature Engineering。S5 把所有技巧串起來，對一個全新資料集做**完整 EDA** → **EDA Capstone**